In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("results/results.csv")
df.head()

In [ ]:
print("Rows:", len(df))
print(df.columns.tolist())
print(df[["model", "temperature", "prompt_version"]].drop_duplicates().sort_values(["model", "temperature", "prompt_version"]))

In [ ]:
key_cols = ["condition", "run_number", "model", "temperature", "prompt_version"]

print("Rows before dedupe:", len(df))
dupes = df.duplicated(subset=key_cols, keep=False)
print("Duplicate rows on run key:", dupes.sum())

df_dedup = df.drop_duplicates(subset=key_cols, keep="first").copy()
print("Rows after dedupe:", len(df_dedup))

In [ ]:
analysis_df = df_dedup[df_dedup["valid"] == True].copy()
analysis_df = analysis_df.dropna(subset=["parsed_wager", "log_wager"])
print("Usable rows:", len(analysis_df))

In [ ]:
block_counts = (
    analysis_df.groupby(["model", "temperature", "prompt_version", "condition"])
    .size()
    .reset_index(name="n")
    .sort_values(["model", "temperature", "prompt_version", "condition"])
)

block_counts

In [ ]:
analysis_df.groupby(["model", "temperature", "prompt_version"]).size()

In [ ]:
def classify_domain(condition):
    return "paper" if condition.startswith("paper") else "realized"

def classify_valence(condition):
    if "gain" in condition:
        return "gain"
    if "neutral" in condition:
        return "neutral"
    return "loss"

analysis_df["domain"] = analysis_df["condition"].apply(classify_domain)
analysis_df["valence"] = analysis_df["condition"].apply(classify_valence)

analysis_df[["condition", "domain", "valence"]].drop_duplicates().sort_values("condition")

In [ ]:
main_table = (
    analysis_df.groupby(["prompt_version", "temperature", "domain", "valence"])
    .agg(
        n=("parsed_wager", "size"),
        mean_wager=("parsed_wager", "mean"),
        median_wager=("parsed_wager", "median"),
        mean_log_wager=("log_wager", "mean")
    )
    .reset_index()
    .sort_values(["prompt_version", "temperature", "domain", "valence"])
)

main_table

In [ ]:
plot_df = (
    analysis_df.groupby(["prompt_version", "temperature", "condition"])
    .agg(mean_wager=("parsed_wager", "mean"))
    .reset_index()
)

for prompt in sorted(plot_df["prompt_version"].unique()):
    for temp in sorted(plot_df["temperature"].unique()):
        temp_df = plot_df[(plot_df["prompt_version"] == prompt) & (plot_df["temperature"] == temp)].copy()
        temp_df = temp_df.sort_values("condition")
        
        plt.figure(figsize=(10, 4))
        plt.bar(temp_df["condition"], temp_df["mean_wager"])
        plt.xticks(rotation=45, ha="right")
        plt.ylabel("Mean wager")
        plt.title(f"Mean wager by condition | prompt={prompt}, temp={temp}")
        plt.tight_layout()
        plt.show()

In [ ]:
paper_vs_realized = (
    analysis_df.groupby(["prompt_version", "temperature", "domain"])
    .agg(
        n=("parsed_wager", "size"),
        mean_wager=("parsed_wager", "mean"),
        mean_log_wager=("log_wager", "mean")
    )
    .reset_index()
)

paper_vs_realized

In [ ]:
!pip install statsmodels

In [ ]:
import statsmodels.formula.api as smf

In [ ]:
model2 = smf.ols(
    "log_wager ~ C(domain) * C(valence) * C(prompt_version) + C(temperature)",
    data=analysis_df
).fit()

print(model2.summary())